# Seasonal dengue: a mosquito–human model fitted with Bayesian MCMC

A mosquito–human transmission model whose transmission rate varies seasonally (Fourier components of the case series),
fitted to monthly case counts with a negative-binomial likelihood and an adaptive random-walk Metropolis sampler.

Runs on synthetic data by default; set `DATA_PATH` in Step 1 to use your own monthly counts.

## Step 0 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
from scipy.stats import nbinom
from scipy.special import expit, logit

## Step 1 — Settings and constants (all rates per month)

In [ ]:
DATA_PATH = None                        # None = synthetic data; or a .xlsx/.csv file with an 'infections' column

DAYS_PER_MONTH = 30.4

N_HUMANS = 1_575_000                    # human population
N_MOSQUITOES = 5e8                      # initial susceptible mosquitoes
INITIAL_INFECTED_MOSQUITOES = 5e5       # initial infected mosquitoes

MU_H = 0.0121 / 12                      # human birth rate
DELTA_H = 0.0059 / 12                   # human death rate
GAMMA_H = DAYS_PER_MONTH / 7            # recovery rate (7-day infection)
MU_M = DAYS_PER_MONTH / 42              # mosquito death rate (42-day lifespan)
MU_B = MU_M                             # mosquito birth rate (constant population)

N_FOURIER = 5                           # number of seasonal components
RANDOM_SEED = 42


## Step 2 — Data (synthetic unless `DATA_PATH` is set)

In [ ]:
if DATA_PATH is None:
    # Synthetic data: 12 years of monthly counts with a monsoon-season peak, year-to-year variation
    # and negative-binomial noise. It only has the right shape; it is not real data.
    gen = np.random.default_rng(RANDOM_SEED)
    n_months = 144
    month = np.arange(n_months)
    seasonal = np.exp(1.6 * np.cos(2 * np.pi * (month % 12 - 8) / 12))
    yearly = np.repeat(gen.lognormal(0.0, 0.6, n_months // 12), 12)
    mean_cases = 6 * seasonal * yearly
    dispersion = 5.0
    cases = gen.negative_binomial(dispersion, dispersion / (dispersion + mean_cases)).astype(int)
    data_label = "Synthetic monthly dengue cases"
else:
    df = pd.read_csv(DATA_PATH) if str(DATA_PATH).endswith(".csv") else pd.read_excel(DATA_PATH)
    df.columns = df.columns.str.strip()
    cases = df["infections"].to_numpy().astype(int)
    n_months = len(cases)
    data_label = "Monthly reported dengue cases"
print(f"{n_months} months of data, total cases = {cases.sum()}, "
      f"mean per month = {cases.mean():.1f}, max = {cases.max()}")

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(np.arange(n_months), cases, "o-", color="black", ms=3)
ax.set_xlabel("Month")
ax.set_ylabel("Reported cases")
ax.set_title(data_label)
ax.grid(alpha=0.3)
plt.show()


## Step 3 — Seasonal cycles from the Fourier spectrum

In [ ]:
def seasonal_factor(t, freqs, a, b):
    angle = 2 * np.pi * freqs * t
    return np.exp(np.dot(a, np.cos(angle)) + np.dot(b, np.sin(angle)))


def find_seasonal_cycles(counts, n_components):
    t = np.arange(len(counts))
    log_counts = np.log(counts + 1)
    centred = log_counts - log_counts.mean()
    power = np.abs(np.fft.rfft(centred)) ** 2
    freqs = np.fft.rfftfreq(len(counts), d=1)
    candidates = np.where(freqs > 0.01)[0]
    order = np.argsort(power[candidates])[::-1]
    chosen = freqs[candidates[order[:n_components]]]

    angle = 2 * np.pi * np.outer(t, chosen)
    X = np.hstack([np.cos(angle), np.sin(angle)])
    coefs = np.linalg.lstsq(X, centred, rcond=None)[0]
    return chosen, coefs[:n_components], coefs[n_components:]


FREQS, A_FIT, B_FIT = find_seasonal_cycles(cases, N_FOURIER)
print("Chosen cycle lengths (months):", np.round(1 / FREQS, 1))

t_fine = np.linspace(0, n_months - 1, 600)
fit_curve = [seasonal_factor(ti, FREQS, A_FIT, B_FIT) for ti in t_fine]
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(np.arange(n_months), np.log(cases + 1) - np.log(cases + 1).mean(), "k.", label="log(cases+1), centred")
ax.plot(t_fine, np.log(fit_curve), color="tab:orange", label="Fourier fit")
ax.set_xlabel("Month")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Step 4 — Transmission model

In [ ]:
def dengue_model(y, t, beta_m, beta_h, freqs, a, b):
    S_h, I_h, R_h, S_m, I_m, C = y
    s = seasonal_factor(t, freqs, a, b)
    N_h = S_h + I_h + R_h
    N_m = S_m + I_m

    new_h = beta_m * s * S_h * I_m / N_m
    new_m = beta_h * s * S_m * I_h / N_h

    dS_h = MU_H * N_h - new_h - DELTA_H * S_h
    dI_h = new_h - (GAMMA_H + DELTA_H) * I_h
    dR_h = GAMMA_H * I_h - DELTA_H * R_h
    dS_m = MU_B * N_m - new_m - MU_M * S_m
    dI_m = new_m - MU_M * I_m
    dC = new_h
    return [dS_h, dI_h, dR_h, dS_m, dI_m, dC]

## Step 5 — Parameters and expected reported cases

In [ ]:
N_PARAMS = 4 + 2 * N_FOURIER
PARAM_NAMES = (["log_beta_m", "log_beta_h", "log_r", "logit_rho"]
               + [f"a{k+1}" for k in range(N_FOURIER)]
               + [f"b{k+1}" for k in range(N_FOURIER)])


def unpack(theta):
    return {
        "beta_m": np.exp(theta[0]),
        "beta_h": np.exp(theta[1]),
        "r": np.exp(theta[2]),
        "rho": expit(theta[3]),
        "a": theta[4:4 + N_FOURIER],
        "b": theta[4 + N_FOURIER:],
    }


def expected_cases(theta):
    p = unpack(theta)
    y0 = [N_HUMANS, 0.0, 0.0, N_MOSQUITOES, INITIAL_INFECTED_MOSQUITOES, 0.0]
    t = np.arange(n_months + 1)
    sol, info = odeint(dengue_model, y0, t,
                       args=(p["beta_m"], p["beta_h"], FREQS, p["a"], p["b"]),
                       full_output=True, mxstep=5000)
    if info["message"] != "Integration successful." or not np.all(np.isfinite(sol)):
        return None
    new_infections = np.maximum(np.diff(sol[:, 5]), 0)
    return p["rho"] * new_infections

## Step 6 — Prior, likelihood, posterior

In [ ]:
PRIOR_MEAN = np.array([0.5, 0.5, np.log(10), logit(0.1)] + [0.0] * (2 * N_FOURIER))
PRIOR_SD = np.array([2.0, 2.0, 1.0, 1.0] + [1.0] * (2 * N_FOURIER))


def log_prior(theta):
    z = (theta - PRIOR_MEAN) / PRIOR_SD
    return -0.5 * np.sum(z ** 2)


def log_likelihood(theta):
    mu = expected_cases(theta)
    if mu is None:
        return -np.inf
    r = np.exp(theta[2])
    p = r / (r + mu + 1e-9)
    return nbinom.logpmf(cases, n=r, p=p).sum()


def log_posterior(theta):
    return log_prior(theta) + log_likelihood(theta)

## Step 7 — Starting point (posterior mode)

In [ ]:
theta_guess = np.concatenate([PRIOR_MEAN[:4], 0.5 * A_FIT, 0.5 * B_FIT])
print("log posterior at the guess:", round(log_posterior(theta_guess), 1))

result = minimize(lambda th: -log_posterior(th), theta_guess,
                  method="Powell", options={"maxfev": 4000})
theta_map = result.x
print("log posterior at the optimum:", round(-result.fun, 1))
for name, value in zip(PARAM_NAMES, theta_map):
    print(f"  {name:>11s} = {value: .3f}")

## Step 8 — Adaptive random-walk Metropolis sampler

In [ ]:
def run_chain(log_post, start, n_burn, n_keep, thin, rng, label=""):
    d = len(start)
    cov = np.eye(d) * 0.05 ** 2
    current = start.copy()
    current_lp = log_post(current)
    history, kept, kept_lp = [], [], []
    n_accepted = 0
    n_total = n_burn + n_keep

    for i in range(n_total):
        proposal = rng.multivariate_normal(current, cov)
        proposal_lp = log_post(proposal)
        accept = np.log(rng.uniform()) < proposal_lp - current_lp
        if accept:
            current, current_lp = proposal, proposal_lp

        if i < n_burn:
            history.append(current.copy())
            if i >= 500 and i % 100 == 0:
                recent = np.array(history[len(history) // 2:])
                cov = (2.38 ** 2 / d) * np.cov(recent.T) + 1e-6 * np.eye(d)
        else:
            n_accepted += accept
            if (i - n_burn) % thin == 0:
                kept.append(current.copy())
                kept_lp.append(current_lp)

        if (i + 1) % (n_total // 10) == 0:
            print(f"  {label} {100 * (i + 1) // n_total}% done")

    acceptance_rate = n_accepted / n_keep
    return np.array(kept), np.array(kept_lp), acceptance_rate

## Step 9 — Run two chains

In [ ]:
N_CHAINS = 2
N_BURN = 5000
N_KEEP = 10000
THIN = 5

rng = np.random.default_rng(RANDOM_SEED)
chain_samples, chain_lp, acceptance = [], [], []
for c in range(N_CHAINS):
    start = theta_map + rng.normal(0, 0.1, size=N_PARAMS)
    s, lp, acc = run_chain(log_posterior, start, N_BURN, N_KEEP, THIN, rng, label=f"chain {c + 1}:")
    chain_samples.append(s)
    chain_lp.append(lp)
    acceptance.append(acc)

chains = np.stack(chain_samples)
chain_lp = np.stack(chain_lp)
samples = chains.reshape(-1, N_PARAMS)
print("shape of pooled samples:", samples.shape)

## Step 10 — Convergence diagnostics (split R̂, acceptance rates, traces)

In [ ]:
def split_rhat(x):
    half = x.shape[1] // 2
    pieces = np.concatenate([x[:, :half], x[:, half:2 * half]])
    n = pieces.shape[1]
    W = pieces.var(axis=1, ddof=1).mean()
    B = n * pieces.mean(axis=1).var(ddof=1)
    var_hat = (n - 1) / n * W + B / n
    return np.sqrt(var_hat / W)


print("Acceptance rates:", [round(a, 3) for a in acceptance])
print(f"\n{'parameter':>11s}   R-hat")
for j, name in enumerate(PARAM_NAMES):
    rhat = split_rhat(chains[:, :, j])
    flag = "" if rhat < 1.01 else ("  <- check" if rhat < 1.05 else "  <- NOT converged")
    print(f"{name:>11s}   {rhat:.3f}{flag}")

fig, axes = plt.subplots(N_PARAMS + 1, 1, figsize=(12, 1.4 * (N_PARAMS + 1)), sharex=True)
for j, name in enumerate(PARAM_NAMES):
    for c in range(N_CHAINS):
        axes[j].plot(chains[c, :, j], lw=0.5, alpha=0.8)
    axes[j].set_ylabel(name, rotation=0, ha="right", fontsize=8)
for c in range(N_CHAINS):
    axes[-1].plot(chain_lp[c], lw=0.5, alpha=0.8)
axes[-1].set_ylabel("log post.", rotation=0, ha="right", fontsize=8)
axes[-1].set_xlabel("kept sample")
plt.tight_layout()
plt.show()

## Step 11 — Posterior summaries

In [ ]:
beta_m = np.exp(samples[:, 0])
beta_h = np.exp(samples[:, 1])
r_disp = np.exp(samples[:, 2])
rho = expit(samples[:, 3])
R0 = np.sqrt(beta_m * beta_h / (MU_M * (GAMMA_H + DELTA_H)))

summary = {
    "beta_m (per month)": beta_m,
    "beta_h (per month)": beta_h,
    "dispersion r": r_disp,
    "reporting rate rho": rho,
    "R0 (average season)": R0,
}
print(f"{'quantity':>22s}   {'median':>9s}   {'90% interval':>22s}")
for name, x in summary.items():
    lo, med, hi = np.percentile(x, [5, 50, 95])
    print(f"{name:>22s}   {med:9.4f}   [{lo:9.4f}, {hi:9.4f}]")

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(np.log(beta_m), np.log(beta_h), s=2, alpha=0.3)
ax.set_xlabel("log beta_m")
ax.set_ylabel("log beta_h")
ax.set_title("Joint posterior of the two transmission rates")
ax.grid(alpha=0.3)
plt.show()

## Step 12 — Posterior predictive check

In [ ]:
idx = rng.choice(len(samples), size=300, replace=False)
sims = []
for k in idx:
    mu = expected_cases(samples[k])
    if mu is None:
        continue
    r = np.exp(samples[k, 2])
    sims.append(rng.negative_binomial(r, r / (r + mu + 1e-9)))
sims = np.array(sims)

q05, q25, q50, q75, q95 = np.percentile(sims, [5, 25, 50, 75, 95], axis=0)
months = np.arange(n_months)
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(months, q05, q95, color="tab:blue", alpha=0.2, label="90% predictive band")
ax.fill_between(months, q25, q75, color="tab:blue", alpha=0.4, label="50% predictive band")
ax.plot(months, q50, color="tab:blue", lw=1, label="predictive median")
ax.plot(months, cases, "ko", ms=3, label="observed")
ax.set_xlabel("Month")
ax.set_ylabel("Reported cases")
ax.set_title("Posterior predictive check")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

inside = np.mean((cases >= q05) & (cases <= q95))
print(f"Share of months inside the 90% band: {inside:.0%} (ideally about 90%)")

## Step 13 — Estimated seasonal pattern

In [ ]:
curves = []
for k in idx:
    p = unpack(samples[k])
    curves.append([seasonal_factor(ti, FREQS, p["a"], p["b"]) for ti in t_fine])
curves = np.array(curves)
lo, med, hi = np.percentile(curves, [5, 50, 95], axis=0)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.fill_between(t_fine, lo, hi, color="tab:green", alpha=0.3, label="90% credible band")
ax.plot(t_fine, med, color="tab:green", label="posterior median")
ax.axhline(1, color="grey", ls="--", lw=1)
ax.set_xlabel("Month")
ax.set_ylabel("seasonal factor s(t)")
ax.set_title("Estimated seasonal transmission multiplier")
ax.legend()
ax.grid(alpha=0.3)
plt.show()